# Bài tập 1 - Neuron và các Lớp (Neurons and Layers)
Trong bài thực hành này, chúng ta sẽ khám phá cơ chế hoạt động bên trong của các neuron/đơn vị (units) và các lớp (layers). Cụ thể, bài lab sẽ liên hệ đến các mô hình hồi quy tuyến tính (regression/linear model) và hồi quy logistic (logistic model). Bài thực hành cũng sẽ giới thiệu về TensorFlow và minh họa cách triển khai các mô hình này trong framework đó.
<figure>
   <img src="./images/C2_W1_NeuronsAndLayers.png"  style="width:540px;height:200px;" >
</figure>

## Các gói thư viện (Packages)
**TensorFlow và Keras**  
TensorFlow là một gói thư viện học máy do Google phát triển. Vào năm 2019, Google đã tích hợp Keras vào TensorFlow và phát hành TensorFlow 2.0. Keras là một framework được phát triển độc lập bởi François Chollet, cung cấp một giao diện đơn giản, tập trung vào các lớp (layer-centric) cho TensorFlow. Lab 1 này sẽ sử dụng Keras. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras import Sequential
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.activations import sigmoid
from lab_utils_common import dlc
from lab_neurons_utils import plt_prob_1d, sigmoidnp, plt_linear, plt_logistic
plt.style.use('./deeplearning.mplstyle')
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## Neuron không có hàm kích hoạt - Mô hình hồi quy tuyến tính

### DataSet
Chúng ta sẽ sử dụng một ví dụ của hồi quy tuyến tính về giá nhà.

In [ ]:
X_train = np.array([[1.0], [2.0]], dtype=np.float32)           #(size in 1000 square feet)
Y_train = np.array([[300.0], [500.0]], dtype=np.float32)       #(price in 1000s of dollars)

fig, ax = plt.subplots(1,1)
ax.scatter(X_train, Y_train, marker='x', c='r', label="Data Points")
ax.legend( fontsize='xx-large')
ax.set_ylabel('Price (in 1000s of dollars)', fontsize='xx-large')
ax.set_xlabel('Size (1000 sqft)', fontsize='xx-large')
plt.show()

### Mô hình Tuyến tính/Hồi quy (Regression/Linear Model) 
Hàm được thực hiện bởi một neuron không có hàm kích hoạt hoàn toàn giống với hồi quy tuyến tính:
$$ f_{\mathbf{w},b}(x^{(i)}) = \mathbf{w}\cdot x^{(i)} + b \tag{1}$$


Chúng ta có thể định nghĩa một lớp với một neuron hoặc đơn vị (unit) và so sánh nó với hàm hồi quy tuyến tính quen thuộc.

In [ ]:
linear_layer = tf.keras.layers.Dense(units=1, activation = 'linear', )

Hãy cùng kiểm tra các trọng số (weights).

In [ ]:
linear_layer.get_weights()

Chưa có trọng số nào vì các trọng số vẫn chưa được khởi tạo (instantiated). Hãy thử chạy mô hình trên một mẫu trong `X_train`. Việc này sẽ kích hoạt quá trình khởi tạo trọng số. Lưu ý, đầu vào cho lớp phải là mảng 2 chiều (2-D), vì vậy chúng ta sẽ định hình lại (reshape) nó.

In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)

Kết quả là một tensor (một tên gọi khác của mảng) có kích thước là (1,1) hay một phần tử duy nhất.   
Bây giờ hãy xem các trọng số (weights) và hệ số chệch (bias). Các trọng số này được khởi tạo ngẫu nhiên thành các số nhỏ và bias theo mặc định được khởi tạo bằng 0.

In [ ]:
w, b= linear_layer.get_weights()
print(f"w = {w}, b={b}")

Một mô hình hồi quy tuyến tính (1) với một đặc trưng đầu vào đơn lẻ sẽ có một trọng số và một bias. Điều này khớp với kích thước của `linear_layer` ở trên của chúng ta.   

Các trọng số được khởi tạo với các giá trị ngẫu nhiên, vì vậy hãy thiết lập chúng thành một số giá trị đã biết.

In [ ]:
set_w = np.array([[200]])
set_b = np.array([100])

# set_weights takes a list of numpy arrays
linear_layer.set_weights([set_w, set_b])
print(linear_layer.get_weights())

Hãy so sánh phương trình (1) với đầu ra của lớp.

In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)
alin = np.dot(set_w,X_train[0].reshape(1,1)) + set_b
print(alin)

Bây giờ, chúng ta có thể sử dụng lớp tuyến tính của mình để đưa ra các dự đoán trên dữ liệu huấn luyện.

In [ ]:
prediction_tf = linear_layer(X_train)
prediction_np = np.dot( X_train, set_w) + set_b

In [ ]:
plt_linear(X_train, Y_train, prediction_tf, prediction_np)

## Neuron với hàm kích hoạt Sigmoid
Hàm được thực hiện bởi một neuron/đơn vị với hàm kích hoạt sigmoid hoàn toàn giống với hồi quy logistic:
$$ f_{\mathbf{w},b}(x^{(i)}) = g(\mathbf{w}x^{(i)} + b) \tag{2}$$
trong đó $$g(x) = sigmoid(x)$$ 

Hãy thiết lập $w$ và $b$ thành một số giá trị đã biết và kiểm tra mô hình.


### DataSet
Chúng ta sẽ sử dụng một ví dụ sau:

In [ ]:
X_train = np.array([0., 1, 2, 3, 4, 5], dtype=np.float32).reshape(-1,1)  # 2-D Matrix
Y_train = np.array([0,  0, 0, 1, 1, 1], dtype=np.float32).reshape(-1,1)  # 2-D Matrix

In [ ]:
pos = Y_train == 1
neg = Y_train == 0

fig,ax = plt.subplots(1,1,figsize=(4,3))
ax.scatter(X_train[pos], Y_train[pos], marker='x', s=80, c = 'red', label="y=1")
ax.scatter(X_train[neg], Y_train[neg], marker='o', s=100, label="y=0", facecolors='none', 
              edgecolors=dlc["dlblue"],lw=3)

ax.set_ylim(-0.08,1.1)
ax.set_ylabel('y', fontsize=12)
ax.set_xlabel('x', fontsize=12)
ax.set_title('one variable plot')
ax.legend(fontsize=12)
plt.show()

### Neuron Logistic
Chúng ta có thể triển khai một 'neuron logistic' bằng cách thêm hàm kích hoạt sigmoid. Khi đó, hàm của neuron được mô tả bởi công thức (2) ở trên.   
Phần này sẽ tạo một Mô hình TensorFlow chứa lớp logistic của chúng ta để minh họa một phương pháp khác nhằm tạo mô hình. TensorFlow thường được sử dụng để tạo các mô hình nhiều lớp. Mô hình [Sequential](https://keras.io/guides/sequential_model/) là một cách thuận tiện để xây dựng các mô hình này.

In [ ]:
model = Sequential(
    [
        tf.keras.layers.Dense(1, input_dim=1,  activation = 'sigmoid', name='L1')
    ]
)

`model.summary()` hiển thị các lớp và số lượng tham số trong mô hình. Chỉ có một lớp duy nhất trong mô hình này và lớp đó chỉ có một đơn vị (unit). Đơn vị này có hai tham số là $w$ và $b$.

In [ ]:
model.summary()

In [ ]:
logistic_layer = model.get_layer('L1')
w,b = logistic_layer.get_weights()
print(w,b)
print(w.shape,b.shape)

Hãy thiết lập trọng số và bias thành một số giá trị đã biết.

In [ ]:
set_w = np.array([[2]])
set_b = np.array([-4.5])
# set_weights takes a list of numpy arrays
logistic_layer.set_weights([set_w, set_b])
print(logistic_layer.get_weights())

Hãy so sánh phương trình (2) với đầu ra của lớp.

In [ ]:
a1 = model.predict(X_train[0].reshape(1,1))
print(a1)
alog = sigmoidnp(np.dot(set_w,X_train[0].reshape(1,1)) + set_b)
print(alog)

Chúng cho ra cùng các giá trị!
Bây giờ, chúng ta có thể sử dụng lớp logistic và mô hình NumPy của mình để đưa ra các dự đoán trên dữ liệu huấn luyện.

In [ ]:
plt_logistic(X_train, Y_train, model, set_w, set_b, pos, neg)

Vùng bóng mờ ở trên phản ánh đầu ra của hàm sigmoid, dao động trong khoảng từ 0 đến 1.

# Chúc mừng bạn!
Bạn đã xây dựng một mạng neural rất đơn giản và đã khám phá được sự tương đồng giữa một neuron với hồi quy tuyến tính và hồi quy logistic.

*Nguồn: Andrew Ng, Advanced Learning Algorithms, DeepLearning.AI & Standford Online.*